<a href="https://colab.research.google.com/github/ofir2207/Cloud-project/blob/main/ex5_cloud.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import firebase_admin
from firebase_admin import credentials

# Initialization code in Colab
if not firebase_admin._apps:
    # Use the JSON file you uploaded to Colab
    cred = credentials.Certificate('plant-disease-project-375ca-firebase-adminsdk-fbsvc-fd7024c7e5.json')

    firebase_admin.initialize_app(cred, {
        # PASTE YOUR URL HERE:
        'databaseURL': 'https://plant-disease-project-375ca-default-rtdb.firebaseio.com/'
    })

In [ ]:
import requests
from bs4 import BeautifulSoup
import collections
import re
import nltk
from nltk.stem import PorterStemmer

# Download necessary resources for NLP
nltk.download('punkt')

# Define the target URL
url = "https://earthsally.com/disease-control/common-plant-diseases.html"

# Define 10 chosen words
my_selected_words = [
    'fungi', 'bacteria', 'virus', 'lesion', 'mold',
    'leaf', 'spot', 'infection', 'rot', 'mildew'
]

def get_index_from_url(url, selected_words):
    try:
        response = requests.get(url)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        return f"Error fetching the URL: {e}"

    soup = BeautifulSoup(response.text, 'html.parser')
    for script_or_style in soup(["script", "style"]):
        script_or_style.decompose()

    text = soup.get_text().lower()

    # NLP Step: Initialize the Porter Stemmer
    stemmer = PorterStemmer()

    # Process the website text
    all_words = re.findall(r'\b[a-z]{3,}\b', text)
    # Stem every word found on the site
    all_stemmed_on_site = [stemmer.stem(w) for w in all_words]

    # Count occurrences of all stems
    counts = collections.Counter(all_stemmed_on_site)

    # Create index only for selected words
    index = {}
    for word in selected_words:
        stemmed_target = stemmer.stem(word.lower())
        index[word] = counts[stemmed_target]

    return index

# Execute and store the index using your list
plant_disease_index = get_index_from_url(url, my_selected_words)

# Display the results
print("Your Selected Keywords and their Counts:")
for word, count in plant_disease_index.items():
    print(f"{word}: {count}")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Your Selected Keywords and their Counts:
fungi: 0
bacteria: 0
virus: 0
lesion: 0
mold: 0
leaf: 5
spot: 13
infection: 11
rot: 2
mildew: 19


In [ ]:
# 1. Verification of Libraries
import firebase_admin
from firebase_admin import credentials, db

# 2. Reset connection if it exists (to avoid 'app already exists' error)
if firebase_admin._apps:
    # Iterate over the App objects (values) instead of the app names (keys)
    for app_object in list(firebase_admin._apps.values()):
        firebase_admin.delete_app(app_object)

print("Starting Firebase connection...")

try:
    # 3. Initialize with your specific data
    # Make sure 'your-key.json' is the EXACT name of the file in your Colab sidebar
    cred = credentials.Certificate('plant-disease-project-375ca-firebase-adminsdk-fbsvc-fd7024c7e5.json')

    firebase_admin.initialize_app(cred, {
        'databaseURL': 'https://plant-disease-project-375ca-default-rtdb.firebaseio.com/'
    })
    print("Connection established!")

    # 4. Reference to your DB location
    ref = db.reference('plant_disease_index')

    # 5. The actual upload
    # 'plant_disease_index' is the dictionary we created in Section A
    ref.set(plant_disease_index)

    print("------------------------------------------")
    print("SUCCESS: Data was sent to Firebase!")
    print("Go check your Firebase Console now.")
    print("------------------------------------------")

except Exception as e:
    print("------------------------------------------")
    print(f"FAILED! The error is: {e}")
    print("Check if your JSON file name or Database URL is correct.")
    print("------------------------------------------")

Starting Firebase connection...
Connection established!
------------------------------------------
SUCCESS: Data was sent to Firebase!
Go check your Firebase Console now.
------------------------------------------
